In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import WavLMModel
from tqdm import tqdm
from torch.cuda.amp import GradScaler

# ==========================================
# 1. DATASET & DATALOADER
# ==========================================
class ASVspoof2019LA(Dataset):
    def __init__(self, protocol_file, audio_dir, max_frames=64000, target_sr=16000):
        self.audio_dir = audio_dir
        self.max_frames = max_frames
        self.target_sr = target_sr
        self.data_list = []
        with open(protocol_file, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    label = 0 if parts[4] == 'bonafide' else 1
                    self.data_list.append((parts[1], label))

    def __len__(self): return len(self.data_list)

    def process_audio(self, waveform, sr):
        if sr != self.target_sr:
            waveform = torchaudio.transforms.Resample(sr, self.target_sr)(waveform)
        num_frames = waveform.shape[1]
        if num_frames > self.max_frames:
            waveform = waveform[:, :self.max_frames]
        elif num_frames < self.max_frames:
            waveform = F.pad(waveform, (0, self.max_frames - num_frames))
        return waveform

    def __getitem__(self, idx):
        audio_id, label = self.data_list[idx]
        file_path = os.path.join(self.audio_dir, f"{audio_id}.flac")
        waveform, sr = torchaudio.load(file_path)
        if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
        waveform = self.process_audio(waveform, sr).squeeze(0)
        return waveform, torch.tensor(label, dtype=torch.long)

# ==========================================
# 2. MODELS (TEACHER & STUDENT)
# ==========================================
class WavLMTeacher(nn.Module):
    def __init__(self, model_name="microsoft/wavlm-base-plus", num_classes=2):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained(model_name)
        self.head = nn.Sequential(
            nn.Linear(self.wavlm.config.hidden_size, 256),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(256, num_classes)
        )
    def forward(self, waveform):
        outputs = self.wavlm(waveform)
        hidden_states = outputs.last_hidden_state
        return self.head(hidden_states.mean(dim=1)), hidden_states

class ModifiedAASIST(nn.Module):
    """
    ĐÂY LÀ MÔ HÌNH GIẢ ĐỊNH. BẠN HÃY THAY THẾ BẰNG CLASS AASIST THẬT CỦA BẠN.
    Yêu cầu: Hàm forward phải trả về (logits, intermediate_features)
    """
    def __init__(self):
        super().__init__()
        # Giả lập SincNet + CNN trả về 128 channels
        self.encoder = nn.Conv1d(1, 128, kernel_size=251, stride=10) 
        self.classifier = nn.Linear(128, 2)
    def forward(self, x):
        x = x.unsqueeze(1) # Thêm channel dim: (Batch, 1, 64000)
        features = self.encoder(x) # Shape: (Batch, 128, Frames)
        pooled = features.mean(dim=2)
        logits = self.classifier(pooled)
        return logits, features

# ==========================================
# 3. FOCUS DISTILLATION LOSS & FGSM
# ==========================================
class FocusDistillationLoss(nn.Module):
    def __init__(self, student_dim=128, teacher_dim=768):
        super().__init__()
        self.projector = nn.Sequential(
            nn.Conv1d(student_dim, teacher_dim, kernel_size=1),
            nn.BatchNorm1d(teacher_dim), nn.ReLU()
        )
        self.mse_loss = nn.MSELoss(reduction='none')

    def generate_focus_mask(self, teacher_features):
        frame_energy = torch.mean(torch.abs(teacher_features), dim=2)
        min_val = frame_energy.min(dim=1, keepdim=True)[0]
        max_val = frame_energy.max(dim=1, keepdim=True)[0]
        mask = (frame_energy - min_val) / (max_val - min_val + 1e-8)
        return mask.unsqueeze(-1)

    def forward(self, student_features, teacher_features):
        # Đặc trưng Student: (Batch, 128, Frames)
        projected = self.projector(student_features)
        projected = F.interpolate(projected, size=teacher_features.size(1), mode='linear', align_corners=False)
        projected = projected.transpose(1, 2) # -> (Batch, Frames, 768)
        
        mask = self.generate_focus_mask(teacher_features)
        loss = self.mse_loss(projected, teacher_features) * mask
        return torch.mean(loss)

def fgsm_attack(model, criterion, waveforms, labels, epsilon=0.005):
    waveforms_adv = waveforms.clone().detach().requires_grad_(True)
    with torch.amp.autocast('cuda'):
        logits, _ = model(waveforms_adv)
        loss = criterion(logits, labels)
    model.zero_grad()
    loss.backward()
    perturbed = waveforms_adv + epsilon * waveforms_adv.grad.data.sign()
    return torch.clamp(perturbed, -1.0, 1.0).detach()

# ==========================================
# 4. CẤU HÌNH & HUẤN LUYỆN (FINAL PIPELINE)
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Bắt đầu huấn luyện trên: {device}")

    # --- ĐƯỜNG DẪN DỮ LIỆU (Cập nhật theo Colab của bạn) ---
    PROTOCOL_PATH = "/content/LA_train_data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
    AUDIO_DIR = "/content/LA_train_data/ASVspoof2019_LA_train/flac/"
    
    # Init DataLoader cho A100
    try:
        train_dataset = ASVspoof2019LA(PROTOCOL_PATH, AUDIO_DIR)
        train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=8, pin_memory=True)
    except:
        print("⚠️ Không tìm thấy dữ liệu! Code sẽ chạy thử với Dummy Data để kiểm tra luồng.")
        # Dummy data để test pipeline nếu chưa có file
        train_loader = [(torch.randn(16, 64000), torch.randint(0, 2, (16,)))]

    # Init Models
    teacher_model = WavLMTeacher().to(device)
    # TẢI TRỌNG SỐ GIÁO VIÊN ĐÃ FINETUNE Ở BƯỚC TRƯỚC
    if os.path.exists("wavlm_teacher_finetuned.pth"):
        teacher_model.load_state_dict(torch.load("wavlm_teacher_finetuned.pth", map_location=device))
        print("✅ Đã tải trọng số Giáo viên WavLM.")
    teacher_model.eval() # Bắt buộc: Đóng băng Teacher
    for param in teacher_model.parameters(): param.requires_grad = False

    student_model = ModifiedAASIST().to(device) # Thay bằng AASIST của bạn
    
    # Init Loss & Optimizer
    task_criterion = nn.CrossEntropyLoss()
    distill_criterion = FocusDistillationLoss(student_dim=128, teacher_dim=768).to(device)
    
    # Train cả Student và Projector của Distillation Loss
    optimizer = optim.AdamW(list(student_model.parameters()) + list(distill_criterion.parameters()), lr=1e-4)
    scaler = GradScaler()

    # Siêu tham số
    epochs = 20
    lambda_weight = 0.5 # Tỷ lệ cân bằng giữa Classification và Distillation
    epsilon = 0.002 # Độ nhiễu FGSM (để nhỏ tránh phá hỏng hoàn toàn âm thanh)

    # --- VÒNG LẶP HUẤN LUYỆN CHÍNH ---
    for epoch in range(epochs):
        student_model.train()
        running_task_loss, running_distill_loss = 0.0, 0.0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for waveforms, labels in progress_bar:
            waveforms, labels = waveforms.to(device), labels.to(device)
            
            # 1. TẠO NHIỄU FGSM TRÊN STUDENT (Adversarial Attack)
            student_model.eval() # Eval mode để sinh nhiễu ổn định
            noisy_waveforms = fgsm_attack(student_model, task_criterion, waveforms, labels, epsilon)
            student_model.train()
            
            # 2. LẤY ĐẶC TRƯNG CHUẨN TỪ TEACHER (Bản Sạch)
            with torch.no_grad():
                with torch.amp.autocast('cuda'):
                    _, teacher_features = teacher_model(waveforms)
            
            # 3. HUẤN LUYỆN STUDENT (Trên Bản Nhiễu)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                student_logits, student_features = student_model(noisy_waveforms)
                
                # Tính toán Loss
                loss_task = task_criterion(student_logits, labels)
                loss_distill = distill_criterion(student_features, teacher_features)
                total_loss = loss_task + lambda_weight * loss_distill
            
            # Cập nhật trọng số
            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            # Thống kê
            running_task_loss += loss_task.item()
            running_distill_loss += loss_distill.item()
            
            progress_bar.set_postfix({
                'TaskLoss': f'{running_task_loss/(progress_bar.n+1):.3f}',
                'DistillLoss': f'{running_distill_loss/(progress_bar.n+1):.3f}'
            })
            
    # Lưu trọng số Student
    torch.save(student_model.state_dict(), "robust_aasist_student.pth")
    print("🎉 Hoàn tất quá trình Focus-Based Distillation!")